<h1 style="text-align:center;">
House_Price_Prediction
</h1>

## Import Libraries

In [114]:
import pandas as pd
import numpy as np

## Import Dataset

In [115]:
dataset = pd.read_csv('data.csv')

In [116]:
print(dataset.shape)
print((dataset["price"] == 0).sum())

(4600, 18)
49


In [117]:
dataset = dataset[dataset["price"] > 0].copy()

In [118]:
print(dataset.shape)

(4551, 18)


In [119]:
dataset.isnull().sum()

date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
street           0
city             0
statezip         0
country          0
dtype: int64

In [120]:
X = dataset.drop(columns=["price", "date", "street", "country"])
y = dataset["price"]

In [121]:
print(X.shape)
print(y.shape)

(4551, 14)
(4551,)


In [122]:
X.dtypes

bedrooms         float64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
city              object
statezip          object
dtype: object

# Encode Categorical Features

In [123]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(
    transformers=[('encoder', OneHotEncoder(), [12, 13])],
    remainder='passthrough'
)
X = ct.fit_transform(X)

## Train-Test Split

In [124]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

# Feature Scaling

In [125]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler(with_mean=False)

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Model Comparison (Cross-validation)


In [126]:
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error

y_train_log = np.log1p(y_train)

In [127]:
from sklearn.model_selection import cross_val_score

models = {
    "Linear": LinearRegression(),
    "SVR": SVR(),
    "RandomForest": RandomForestRegressor(random_state=0),
    "XGBoost": XGBRegressor(random_state=0),
}

for name, m in models.items():
    scores = cross_val_score(m, X_train, y_train_log, cv=5, scoring="r2")
    print(name, "R2 (log):", round(scores.mean(), 4), "+/-", round(scores.std(), 4))

Linear R2 (log): 0.7697 +/- 0.0287
SVR R2 (log): 0.768 +/- 0.0325
RandomForest R2 (log): 0.7147 +/- 0.032
XGBoost R2 (log): 0.7437 +/- 0.0353


# SVR Tuning & Final Evaluation

In [128]:
from sklearn.model_selection import GridSearchCV

param_grid = {"C": [1, 10, 100], "epsilon": [0.01, 0.1]}

grid = GridSearchCV(SVR(), param_grid, cv=5, scoring="r2")
grid.fit(X_train, y_train_log)
print(grid.best_params_, grid.best_score_)

best_svr = grid.best_estimator_
y_pred_best = np.expm1(best_svr.predict(X_test))
print("R2:", r2_score(y_test, y_pred_best))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_best)))

{'C': 10, 'epsilon': 0.1} 0.7775923418990411
R2: 0.793099901112917
RMSE: 166597.73508470875


# Save the Model

In [129]:
import joblib

joblib.dump(ct, "ct.joblib")
joblib.dump(sc, "scaler.joblib")
joblib.dump(best_svr, "svr_model.joblib")